<a href="https://colab.research.google.com/github/akashvoffi-design/WorkSpace-ML/blob/main/Email_Spam_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Install & Import Libraries
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [3]:
#Load the Dataset
from google.colab import files
uploaded = files.upload()   # select spam.csv

df = pd.read_csv('spam.csv', encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
df.head()

Saving spam.csv to spam (1).csv


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
#Clean & Explore
df = df.drop_duplicates(keep='first').reset_index(drop=True)
print(df['label'].value_counts())


label
ham     4516
spam     653
Name: count, dtype: int64


In [5]:
#Text Preprocessing (the NLP core)
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def transform_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [ps.stem(w) for w in words]
    return " ".join(words)

df['transformed_message'] = df['message'].apply(transform_text)

In [6]:
#TF-IDF Feature Extraction
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df['transformed_message']).toarray()
y = df['label'].map({'ham': 0, 'spam': 1}).values

In [7]:
#Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [8]:
#Train All 3 Models
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(kernel='sigmoid', gamma=1.0, probability=True)
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1-score": f1_score(y_test, preds)
    })
    trained_models[name] = model

pd.DataFrame(results)

,Model,Accuracy,Precision,Recall,F1-score
0,Naive Bayes,0.976789,0.981982,0.832061,0.900826
1,Logistic Regression,0.958414,1.000000,0.671756,0.803653
2,SVM,0.975822,0.973214,0.832061,0.897119


In [9]:
#Real-Time Prediction Function
best_model = trained_models["Naive Bayes"]

def predict_message(text, model=best_model, vectorizer=tfidf):
    cleaned = transform_text(text)
    vec = vectorizer.transform([cleaned]).toarray()
    pred = model.predict(vec)[0]
    return "SPAM 🚫" if pred == 1 else "HAM ✅"

predict_message("Congratulations! You won a free gift card, click now!!!")

'SPAM 🚫'

In [10]:
predict_message("IV data details of the student and teachers")

'HAM ✅'